In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Load model
model = joblib.load("models/saved_models/RF_model.pkl")

# Load dataset (same as existing notebook)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# Split (same parameters as existing)
x = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Generate report
accuracy = model.score(X_test, y_test)
roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovo')

print("=" * 60)
print("RANDOM FOREST AUDIT REPORT")
print("=" * 60)

print(f"\nModel: {type(model).__name__}")
print(f"Trees: {model.n_estimators}")
print(f"\nAccuracy: {accuracy:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
target_names = data.target_names
print(classification_report(y_test, y_pred, target_names=target_names))

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()

print("\n" + "=" * 60)
print("TOP 10 FEATURE IMPORTANCE")
print("=" * 60)
importance_df = pd.DataFrame({
    'feature': data.feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df.head(10).to_string(index=False))

# Plot top 10 features
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', 
            data=importance_df.head(10),
            palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()